# Data Science Programming - Sesion 3 (via avanzada)

**Especializacion en Ciencia de Datos - Universidad Santo Tomas - Tunja**
**Prof. Hugo Hernandez | Martes 8 de septiembre, 2026**

---

## Antes de empezar, lee esto

Este notebook **reemplaza** al de la sesion. Si ya programas, trabaja aqui a tu ritmo
mientras el resto del grupo ve los fundamentos de pandas.

- **No vale nota.** Existe para que no te toque esperar.
- **Trabajas solo.** Durante la clase estare con quienes estan empezando, asi que aqui
  tienes todo lo que necesitas. Cada reto se verifica solo: corres la celda de
  verificacion y sabes si lo lograste.
- **No tienes que entregarlo.** Estos retos no se califican. Subelo al aula solo si quieres que lo revise y te comente por escrito: es totalmente opcional y no hay que exponer.
- Si un reto se te atraviesa, sigue con el siguiente y vuelve despues.

La clase normal ve hoy: `DataFrame` y `Series`, el indice, seleccion, `.loc` / `.iloc`,
filtrado booleano, `groupby`, ordenar y `value_counts`. Todo eso lo das por visto. Aqui
vas por lo que de verdad separa a quien sabe pandas de quien solo lo ha usado:
agregacion multiple, joins, `transform`, `pivot_table` y ventanas moviles.

No se descarga nada. Los datos se construyen en el notebook.

---
## Repaso relampago

Si algo de la primera tabla no te suena, mejor toma el notebook normal.

| Operacion | Sintaxis | Nota |
|---|---|---|
| Columna | `df['col']` -> Series; `df[['a','b']]` -> DataFrame | Un corchete vs dos |
| Por etiqueta | `df.loc[fila, col]` | El slice de `.loc` **incluye** el extremo |
| Por posicion | `df.iloc[i, j]` | El slice de `.iloc` excluye el extremo, como Python |
| Filtro | `df[df['x'] > 3]` | Varias condiciones: `&`, `|`, `~` y **parentesis** |
| Agrupar | `df.groupby('k')['v'].sum()` | La clave se vuelve el indice |
| Ordenar | `df.sort_values('v', ascending=False)` | `sort_index()` ordena por indice |
| Contar | `df['c'].value_counts()` | Devuelve Series ordenada desc, nombre `count` |
| Faltantes | `isna()`, `fillna()`, `dropna()` | `NaN != NaN`; nunca compares con `==` |

Lo de hoy, que la clase normal no alcanza:

| Operacion | Para que | Forma tipica |
|---|---|---|
| `.agg` con nombres | Varias metricas de una | `g.agg(tot=('v','sum'), n=('v','size'))` |
| `merge` | Cruzar dos tablas | `a.merge(b, on='k', how='left')` |
| `transform` | Devolver un valor **por fila** calculado sobre su grupo | `g['v'].transform('sum')` |
| `rank` | Posicion dentro del grupo | `.rank(method='dense', ascending=False)` |
| `pivot_table` | Cruce filas x columnas con agregacion | `pivot_table(index=, columns=, values=, aggfunc=)` |
| `resample` | Reagrupar una serie de tiempo | `df.set_index('fecha').resample('D')` |
| `rolling` | Ventana movil | `s.rolling(7, min_periods=7).mean()` |

**La distincion que mas cuesta:** `agg` **reduce** (un grupo -> una fila) y `transform`
**conserva la forma** (un grupo -> tantas filas como entraron). Cuando necesitas comparar
cada fila contra el total de su grupo, es `transform`, no `agg` seguido de `merge`.

---
## Nota de version

Este notebook esta escrito para correr igual en pandas 2.x y 3.x. Dos cosas cambiaron
entre versiones y por eso **no se usan en las verificaciones**:

- El dtype de las columnas de texto (`object` en 2.x, `str` en 3.x) y la resolucion de
  las fechas (`datetime64[ns]` vs `datetime64[us]`). Ningun assert compara dtypes de
  texto ni de fecha por su nombre.
- El comportamiento de copia (Copy-on-Write es el default en 3.x). Por eso los retos
  piden explicitamente **devolver un DataFrame nuevo** en lugar de modificar el que
  reciben: eso es correcto en las dos versiones y ademas es mejor diseno.

Si algo te falla por version, corre `pd.__version__` y avisame.

---
## El dominio de trabajo

Distribuidora de abarrotes en Boyaca. Dos tablas:

- `VENTAS`: un renglon por pedido despachado entre el 1 de junio y el 31 de julio de 2026.
- `CLIENTES`: el maestro de clientes.

Las dos tablas **no casan perfecto**, y eso es a proposito: hay pedidos de clientes que
no estan en el maestro (los cargo un vendedor nuevo) y clientes del maestro que no
compraron nada en el periodo. Ese desajuste es el corazon del reto 2.

Ejecuta esta celda antes de empezar.

In [ ]:
import numpy as np
import pandas as pd

print('pandas', pd.__version__, '| numpy', np.__version__)

CATALOGO = {
    'Bebidas': [('BEB-01', 'Gaseosa 1.5L', 4200), ('BEB-02', 'Agua 600ml', 1800),
                ('BEB-03', 'Jugo caja 1L', 5600), ('BEB-04', 'Energizante', 7400)],
    'Aseo':    [('ASE-01', 'Detergente 1kg', 9800), ('ASE-02', 'Jabon barra', 2300),
                ('ASE-03', 'Blanqueador 1L', 5100)],
    'Snacks':  [('SNK-01', 'Papas 105g', 3900), ('SNK-02', 'Galleta wafer', 1600),
                ('SNK-03', 'Mani salado', 2900), ('SNK-04', 'Chocolatina', 1500)],
    'Lacteos': [('LAC-01', 'Leche 1L', 4900), ('LAC-02', 'Yogur 1L', 7200),
                ('LAC-03', 'Queso 250g', 11500)],
}

CLIENTES = pd.DataFrame([
    ('C01', 'Tienda La Esquina',   'Tunja',    'Tienda'),
    ('C02', 'Supermercado Norte',  'Tunja',    'Supermercado'),
    ('C03', 'Minimercado El Sol',  'Duitama',  'Tienda'),
    ('C04', 'Autoservicio Centro', 'Sogamoso', 'Supermercado'),
    ('C05', 'Cafeteria UPTC',      'Tunja',    'Institucional'),
    ('C06', 'Tienda Don Jose',     'Paipa',    'Tienda'),
    ('C07', 'Mercado Boyaca',      'Duitama',  'Supermercado'),
    ('C08', 'Hotel Hunza',         'Tunja',    'Institucional'),
    ('C09', 'Tienda La 15',        'Sogamoso', 'Tienda'),
    ('C10', 'Colegio San Luis',    'Paipa',    'Institucional'),
    ('C11', 'Distri Express',      'Tunja',    'Supermercado'),
    ('C12', 'Panaderia Trigal',    'Duitama',  'Tienda'),
], columns=['id_cliente', 'nombre', 'ciudad', 'segmento'])

# --- generacion determinista de los pedidos (misma salida en cualquier maquina) ---
rng = np.random.default_rng(20260908)
COMPRADORES = ['C01','C02','C03','C04','C05','C06','C07','C08','C09','C13','C14']
PESOS = np.array([12, 14, 9, 11, 7, 8, 10, 6, 9, 4, 4], dtype=float)
PESOS = PESOS / PESOS.sum()
CANALES = ['Mostrador', 'Domicilio', 'Mayorista']
PLANO = [(lin, sku, nom, pr) for lin, its in CATALOGO.items() for sku, nom, pr in its]

DIAS = pd.date_range('2026-06-01', '2026-07-31', freq='D')
SIN_OPERACION = set(pd.to_datetime([
    '2026-06-15', '2026-06-16', '2026-07-05', '2026-07-06',
    '2026-07-20', '2026-07-21', '2026-07-22']))

_filas, _pid = [], 1000
for _d in DIAS:
    if _d in SIN_OPERACION:
        continue
    for _ in range(int(rng.integers(2, 6))):
        _pid += 1
        _cli = str(rng.choice(COMPRADORES, p=PESOS))
        _lin, _sku, _nom, _pr = PLANO[int(rng.integers(0, len(PLANO)))]
        _can = CANALES[int(rng.integers(0, 3))]
        if _lin == 'Lacteos' and _can == 'Mayorista':
            _can = 'Mostrador'          # cadena de frio: lacteos no salen por mayorista
        _u = int(rng.integers(20, 120) if _can == 'Mayorista' else rng.integers(1, 25))
        _filas.append((_pid, _d, _cli, _lin, _sku, _nom, _u, _pr, _can))

VENTAS = pd.DataFrame(_filas, columns=['id_pedido', 'fecha', 'id_cliente', 'linea',
                                       'sku', 'producto', 'unidades',
                                       'precio_unitario', 'canal'])
VENTAS['importe'] = VENTAS['unidades'] * VENTAS['precio_unitario']

print(f'VENTAS  : {VENTAS.shape[0]} pedidos x {VENTAS.shape[1]} columnas')
print(f'CLIENTES: {CLIENTES.shape[0]} registros')
print(f'periodo : {VENTAS["fecha"].min().date()} a {VENTAS["fecha"].max().date()} '
      f'({VENTAS["fecha"].nunique()} dias con operacion)')
VENTAS.head()

In [ ]:
# El ritual de apertura, por si quieres mirar antes de tocar.
# describe() se aplica a las columnas numericas explicitas: el argumento numeric_only
# existia en pandas 2.x y desaparecio en 3.x. Seleccionar a mano corre en las dos.
VENTAS.info()
print()
print(VENTAS[['unidades', 'precio_unitario', 'importe']].describe().round(1))

---
## Reto 1 - Agregacion multiple por linea

El `groupby` de una sola metrica ya lo sabes. Esto es lo que se pide en el trabajo real:
varias metricas distintas en una sola pasada y con los nombres de columna que uno decide.

Escribe `resumen_lineas(ventas)`.

**Contrato:**
- Recibe un DataFrame con la estructura de `VENTAS`.
- Devuelve un **DataFrame** cuyo indice son las lineas de producto, con `index.name`
  igual a `'linea'`.
- Tiene **exactamente estas cinco columnas, en este orden**:

| Columna | Valor |
|---|---|
| `pedidos` | cuantos renglones tiene la linea |
| `unidades` | suma de `unidades` |
| `importe_total` | suma de `importe` |
| `ticket_medio` | promedio de `importe`, **redondeado a 2 decimales** |
| `clientes_unicos` | cuantos `id_cliente` distintos le compraron |

- Ordenado por `importe_total` **descendente**.
- No debe modificar el DataFrame que recibe.

Hazlo en **un solo `.agg`**. Si te ves calculando cinco `groupby` y pegandolos con
`concat`, te fuiste por el camino largo. Busca "named aggregation" en la documentacion.

In [ ]:
def resumen_lineas(ventas):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 1 ---
_cols_antes = list(VENTAS.columns)
r1 = resumen_lineas(VENTAS)

assert isinstance(r1, pd.DataFrame), 'Debe devolver un DataFrame, no una Series ni un dict'
assert list(VENTAS.columns) == _cols_antes, 'No debe modificar el DataFrame que recibe'
assert r1.index.name == 'linea', \
    f'El indice debe llamarse "linea" y se llama {r1.index.name!r} (mira as_index / reset_index)'
assert sorted(r1.index) == ['Aseo', 'Bebidas', 'Lacteos', 'Snacks'], \
    'El indice debe tener las cuatro lineas de producto'
assert list(r1.columns) == ['pedidos', 'unidades', 'importe_total',
                            'ticket_medio', 'clientes_unicos'], \
    f'Columnas y orden exactos del contrato. Tienes: {list(r1.columns)}'

# expectativas calculadas con Python puro, sin pandas, para que el chequeo sea real
_esp = {}
for _t in VENTAS.itertuples(index=False):
    d = _esp.setdefault(_t.linea, {'n': 0, 'u': 0, 'imp': 0, 'cli': set()})
    d['n'] += 1; d['u'] += _t.unidades; d['imp'] += _t.importe; d['cli'].add(_t.id_cliente)

for _lin, _d in _esp.items():
    assert int(r1.loc[_lin, 'pedidos']) == _d['n'], \
        f'{_lin}: pedidos debe ser {_d["n"]} (numero de renglones), no {r1.loc[_lin, "pedidos"]}'
    assert int(r1.loc[_lin, 'unidades']) == _d['u'], \
        f'{_lin}: unidades debe ser la SUMA de unidades = {_d["u"]}'
    assert int(r1.loc[_lin, 'importe_total']) == _d['imp'], \
        f'{_lin}: importe_total debe ser {_d["imp"]}'
    assert abs(float(r1.loc[_lin, 'ticket_medio']) - round(_d['imp'] / _d['n'], 2)) < 1e-6, \
        f'{_lin}: ticket_medio = promedio de importe redondeado a 2 decimales'
    assert int(r1.loc[_lin, 'clientes_unicos']) == len(_d['cli']), \
        f'{_lin}: clientes_unicos es un conteo de DISTINTOS ({len(_d["cli"])}), no un count'

_v = r1['importe_total'].tolist()
assert _v == sorted(_v, reverse=True), 'Debe quedar ordenado por importe_total descendente'
assert r1.index[0] == 'Bebidas', \
    f'La linea de mayor importe es Bebidas y quedo primera {r1.index[0]!r}'

print('Reto 1 resuelto.')
r1

> **Si te sobra tiempo:** agrega un parametro `por=` que permita agrupar por cualquier
> columna (`'canal'`, `'sku'`, `'id_cliente'`) sin cambiar nada mas, y una columna
> `pct_importe` con la participacion de cada grupo sobre el total.

---
## Reto 2 - El cruce que no casa por ninguno de los dos lados

`VENTAS` y `CLIENTES` comparten `id_cliente`, pero no se corresponden uno a uno. Hay
pedidos cuyo cliente no esta en el maestro y clientes del maestro que no compraron.

El punto del reto no es escribir un `merge`: es que **el numero de filas del resultado
te diga que paso**. Si no sabes explicar por que `inner`, `left` y `outer` dan tres
numeros distintos, no sabes hacer joins.

Escribe `cruce_clientes(ventas, clientes)`.

**Contrato:** devuelve un **diccionario** con exactamente estas seis claves:

| Clave | Valor |
|---|---|
| `'n_inner'` | filas del merge `how='inner'` sobre `id_cliente` |
| `'n_left'` | filas del merge `how='left'` con **`ventas` como tabla izquierda** |
| `'n_outer'` | filas del merge `how='outer'` |
| `'sin_registro'` | lista **ordenada** de los `id_cliente` que aparecen en `ventas` y no estan en `clientes` |
| `'sin_ventas'` | lista **ordenada** de los `id_cliente` que estan en `clientes` y no aparecen en `ventas` |
| `'importe_no_atribuible'` | suma de `importe` de los pedidos cuyo cliente no esta en el maestro (int) |

Los tres conteos deben ser `int` de Python, y las dos listas, listas de `str` de Python
(nada de `np.str_` ni de `Index`; `sorted(...)` sobre valores ya convertidos resuelve).

**Antes de correr la verificacion, escribe en un comentario cuanto crees que va a dar
cada conteo.** Si aciertas los tres, entendiste el join. La relacion no es casual:
`n_outer = n_left + len(sin_ventas)` y `n_inner = n_left - (pedidos huerfanos)`.

In [ ]:
def cruce_clientes(ventas, clientes):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 2 ---
r2 = cruce_clientes(VENTAS, CLIENTES)

assert isinstance(r2, dict), 'Debe devolver un diccionario'
assert set(r2) == {'n_inner', 'n_left', 'n_outer', 'sin_registro',
                   'sin_ventas', 'importe_no_atribuible'}, \
    f'Las seis claves del contrato, ni mas ni menos. Tienes: {sorted(r2)}'

_maestro = set(CLIENTES['id_cliente'])
_en_ventas = set(VENTAS['id_cliente'])
_huerfanas = [t for t in VENTAS.itertuples(index=False) if t.id_cliente not in _maestro]

assert r2['sin_registro'] == sorted(str(x) for x in _en_ventas - _maestro), \
    f'sin_registro son los clientes de VENTAS que faltan en el maestro: ' \
    f'{sorted(_en_ventas - _maestro)}'
assert r2['sin_ventas'] == sorted(str(x) for x in _maestro - _en_ventas), \
    f'sin_ventas son los del maestro que no compraron: {sorted(_maestro - _en_ventas)}'
assert all(isinstance(x, str) for x in r2['sin_registro'] + r2['sin_ventas']), \
    'Las listas deben ser de str de Python; convierte con str(x) o con .tolist()'

assert r2['n_left'] == len(VENTAS), \
    f'n_left debe ser {len(VENTAS)}: el left conserva TODAS las filas de ventas ' \
    f'(el maestro no tiene ids repetidos, asi que no se multiplican filas)'
assert r2['n_inner'] == len(VENTAS) - len(_huerfanas), \
    f'n_inner debe ser {len(VENTAS) - len(_huerfanas)}: el inner descarta los ' \
    f'{len(_huerfanas)} pedidos sin cliente en el maestro'
assert r2['n_outer'] == r2['n_left'] + len(_maestro - _en_ventas), \
    'n_outer = n_left mas una fila por cada cliente del maestro que no compro'
assert r2['n_inner'] < r2['n_left'] < r2['n_outer'], \
    'Si los tres conteos te dan igual, revisa la direccion del merge'
assert all(isinstance(r2[k], int) and not isinstance(r2[k], bool)
           for k in ('n_inner', 'n_left', 'n_outer')), \
    'Los conteos deben ser int de Python (int(...) sobre el len o el shape)'

assert int(r2['importe_no_atribuible']) == sum(t.importe for t in _huerfanas), \
    f'importe_no_atribuible = {sum(t.importe for t in _huerfanas)}, ' \
    f'la plata de los pedidos que el inner bota'

print('Reto 2 resuelto.')
print(f"inner {r2['n_inner']} | left {r2['n_left']} | outer {r2['n_outer']}")
print(f"sin registro en el maestro: {r2['sin_registro']} "
      f"-> {r2['importe_no_atribuible']:,} pesos sin dueno")
print(f"clientes que no compraron : {r2['sin_ventas']}")

> **Si te sobra tiempo:** usa `merge(..., indicator=True)` y saca el mismo diagnostico
> con un `value_counts()` de la columna `_merge`. Es la forma corta y la que se usa en
> produccion para auditar un cruce.

---
## Reto 3 - Comparar cada fila contra su grupo

Este es el reto largo: dos funciones. La primera es la que separa a quien sabe pandas de
quien no.

### 3a. `con_participacion(ventas)`

**Contrato:**
- Devuelve un **DataFrame nuevo** con las mismas filas, **en el mismo orden y con el
  mismo indice** que el que recibe, mas exactamente dos columnas al final:

| Columna | Valor |
|---|---|
| `pct_linea` | `importe` del pedido sobre el `importe` total de **su linea**, en porcentaje (0 a 100), redondeado a 2 decimales |
| `rank_linea` | posicion del pedido dentro de **su linea** por `importe` descendente. 1 es el mayor. Empates comparten posicion y **no se salta** la siguiente (`method='dense'`). Debe ser de tipo **entero** |

- El DataFrame recibido **no se modifica**.

La trampa: `groupby('linea')['importe'].sum()` te da 4 numeros y tu necesitas 196 filas.
Si tu instinto fue agregar y despues hacer `merge` contra el resultado, funciona pero es
el camino largo. `transform` existe justo para esto: devuelve el resultado del grupo
**alineado a cada fila original**.

Ojo con el tipo de `rank_linea`: `rank` devuelve float. El contrato pide entero.

### 3b. `tabla_canal_linea(ventas)`

**Contrato:**
- Devuelve un DataFrame con el `importe` total cruzando **linea** (filas) contra
  **canal** (columnas).
- Indice: las lineas en orden alfabetico, `index.name == 'linea'`.
- Columnas, en este orden exacto: `['Domicilio', 'Mayorista', 'Mostrador', 'total']`,
  donde `total` es la suma de la fila. El eje de columnas **no debe tener nombre**
  (`columns.name is None`; el `pivot_table` se lo pone y hay que quitarlo).
- Hay un cruce que no existe en los datos. Ese hueco debe quedar en **0**, no en `NaN`.
- **Todos los valores deben ser de tipo entero.** Un `fillna(0)` deja la columna en
  float; el parametro correcto de `pivot_table` la deja en entero. Esa es la diferencia
  que se esta evaluando.

In [ ]:
def con_participacion(ventas):
    # tu codigo aqui
    pass


def tabla_canal_linea(ventas):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 3 ---
_cols_antes = list(VENTAS.columns)
r3 = con_participacion(VENTAS)

assert isinstance(r3, pd.DataFrame), '3a debe devolver un DataFrame'
assert list(VENTAS.columns) == _cols_antes, \
    '3a no debe agregarle columnas al DataFrame original (devuelve una copia)'
assert list(r3.columns) == _cols_antes + ['pct_linea', 'rank_linea'], \
    f'Las dos columnas nuevas van al final y se llaman pct_linea y rank_linea. ' \
    f'Tienes: {list(r3.columns)}'
assert len(r3) == len(VENTAS), \
    f'3a conserva las {len(VENTAS)} filas; si te quedaron {len(r3)} agregaste en vez de transformar'
assert r3.index.equals(VENTAS.index), 'El indice debe ser el mismo del DataFrame original'
assert r3['id_pedido'].tolist() == VENTAS['id_pedido'].tolist(), \
    'Las filas deben quedar en el mismo orden (no ordenes por importe)'

_tot = {}
for _t in VENTAS.itertuples(index=False):
    _tot[_t.linea] = _tot.get(_t.linea, 0) + _t.importe
for _i in (0, 7, 55, len(VENTAS) - 1):
    _f = VENTAS.iloc[_i]
    _esperado = round(_f['importe'] / _tot[_f['linea']] * 100, 2)
    assert abs(float(r3['pct_linea'].iloc[_i]) - _esperado) < 1e-6, \
        f'Fila {_i} ({_f["linea"]}): pct_linea deberia ser {_esperado} y es ' \
        f'{r3["pct_linea"].iloc[_i]}. Es porcentaje 0-100 sobre el total de SU linea'
for _lin, _g in r3.groupby('linea'):
    assert abs(_g['pct_linea'].sum() - 100) < 0.5, \
        f'{_lin}: los pct_linea de una misma linea deben sumar 100 (tolerando redondeo), ' \
        f'suman {_g["pct_linea"].sum():.2f}'

assert pd.api.types.is_integer_dtype(r3['rank_linea']), \
    f'rank_linea debe ser entero y quedo en {r3["rank_linea"].dtype} (rank devuelve float)'
for _lin, _g in r3.groupby('linea'):
    assert _g['rank_linea'].min() == 1, f'{_lin}: el pedido mas grande debe tener rank 1'
    assert _g['rank_linea'].max() == _g['importe'].nunique(), \
        f'{_lin}: con method="dense" el rank maximo es el numero de importes DISTINTOS ' \
        f'({_g["importe"].nunique()}), y te dio {_g["rank_linea"].max()}'
    _top = _g.sort_values('rank_linea').iloc[0]
    assert _top['importe'] == _g['importe'].max(), \
        f'{_lin}: rank 1 debe ser el importe MAYOR (ascending=False)'

# --- 3b ---
r3b = tabla_canal_linea(VENTAS)

assert isinstance(r3b, pd.DataFrame), '3b debe devolver un DataFrame'
assert list(r3b.index) == ['Aseo', 'Bebidas', 'Lacteos', 'Snacks'], \
    f'Filas: las cuatro lineas en orden alfabetico. Tienes {list(r3b.index)}'
assert r3b.index.name == 'linea', f'index.name debe ser "linea", es {r3b.index.name!r}'
assert list(r3b.columns) == ['Domicilio', 'Mayorista', 'Mostrador', 'total'], \
    f'Columnas exactas y en ese orden. Tienes: {list(r3b.columns)}'
assert r3b.columns.name is None, \
    'El eje de columnas no debe tener nombre: pivot_table lo deja en "canal", ' \
    'quitalo con rename_axis(columns=None)'
assert not r3b.isna().any().any(), \
    'Quedaron NaN: el cruce que no existe en los datos debe salir en 0'
assert all(pd.api.types.is_integer_dtype(r3b[c]) for c in r3b.columns), \
    f'Todos los valores deben ser enteros. dtypes: {dict(r3b.dtypes)}. ' \
    'fillna(0) deja float; usa fill_value en pivot_table'

_cruce = {}
for _t in VENTAS.itertuples(index=False):
    _cruce[(_t.linea, _t.canal)] = _cruce.get((_t.linea, _t.canal), 0) + _t.importe
assert int(r3b.loc['Lacteos', 'Mayorista']) == 0, \
    'Lacteos nunca se despacho por Mayorista: esa celda debe ser 0'
assert int(r3b.loc['Aseo', 'Domicilio']) == _cruce[('Aseo', 'Domicilio')], \
    f'Aseo por Domicilio deberia ser {_cruce[("Aseo", "Domicilio")]}'
assert int(r3b.loc['Snacks', 'total']) == sum(
    v for (l, _c), v in _cruce.items() if l == 'Snacks'), 'total es la suma de la fila'
assert int(r3b['total'].sum()) == int(VENTAS['importe'].sum()), \
    f'La suma de la columna total debe dar el importe global: {VENTAS["importe"].sum()}'

print('Reto 3 resuelto.')
print(r3[['id_pedido', 'linea', 'importe', 'pct_linea', 'rank_linea']].head(6))
print()
print(r3b)

> **Si te sobra tiempo:** dale la vuelta a `tabla_canal_linea` con `melt` y recupera un
> DataFrame largo de tres columnas (`linea`, `canal`, `importe`). Compara cuantas filas
> te da contra las 12 combinaciones posibles y contra las 11 que existen de verdad: ahi
> se ve que el 0 que metiste es informacion inventada, y en un pipeline real hay que
> decidir si se conserva o se descarta.

---
## Reto 4 - Serie diaria, ventana movil y dias atipicos

El mas duro. Calcula media hora e implica leer documentacion.

La distribuidora no opera todos los dias: hay 7 fechas del calendario sin un solo pedido.
Un `groupby('fecha')` te devuelve **solo los dias con ventas**, y una media movil de 7
dias sobre esa serie estaria promediando ventanas de duracion variable: el resultado es
falso. Hay que **completar el calendario primero**.

### 4a. `serie_diaria(ventas)`

**Contrato:**
- Devuelve un DataFrame indexado por fecha, con **una fila por dia calendario** entre el
  minimo y el maximo de `ventas['fecha']`, sin huecos. `index.name == 'fecha'`.
- Columnas, en este orden:

| Columna | Valor |
|---|---|
| `importe` | suma de `importe` de ese dia; **0** si no hubo operacion |
| `pedidos` | numero de pedidos de ese dia; **0** si no hubo operacion |
| `media7` | media movil de `importe` de los **ultimos 7 dias**, incluyendo el dia actual. Los primeros 6 dias quedan en `NaN` (no hay ventana completa) |
| `indice` | `importe / media7`, redondeado a **3 decimales**. `NaN` donde `media7` sea `NaN` |

- `pedidos` debe ser de tipo entero.

Los `NaN` de `media7` son correctos y **no se rellenan**: son la marca de que ahi no hay
suficiente historia. Rellenarlos con la media de lo que haya es exactamente el error que
la Actividad 2 va a castigar.

### 4b. `dias_pico(serie, umbral)`

**Contrato:**
- Recibe el DataFrame que produce `serie_diaria` y un `umbral` float.
- Devuelve la **lista de fechas** con `indice > umbral`, en orden cronologico ascendente,
  cada una como `str` en formato `'YYYY-MM-DD'`.
- Los dias con `indice` `NaN` nunca entran. No los filtres a mano: piensa que devuelve
  una comparacion contra `NaN`.

In [ ]:
def serie_diaria(ventas):
    # tu codigo aqui
    pass


def dias_pico(serie, umbral):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 4 ---
r4 = serie_diaria(VENTAS)

assert isinstance(r4, pd.DataFrame), '4a debe devolver un DataFrame'
assert isinstance(r4.index, pd.DatetimeIndex), \
    f'El indice debe ser de fechas (DatetimeIndex) y es {type(r4.index).__name__}'
assert r4.index.name == 'fecha', f'index.name debe ser "fecha", es {r4.index.name!r}'
assert list(r4.columns) == ['importe', 'pedidos', 'media7', 'indice'], \
    f'Columnas exactas y en ese orden. Tienes: {list(r4.columns)}'

_ini, _fin = VENTAS['fecha'].min(), VENTAS['fecha'].max()
_n_dias = (_fin - _ini).days + 1
assert len(r4) == _n_dias, \
    f'Deben ser {_n_dias} filas, una por dia calendario. Te dieron {len(r4)}: ' \
    f'si te dio {VENTAS["fecha"].nunique()} agrupaste sin completar el calendario'
assert r4.index.is_monotonic_increasing, 'La serie debe quedar en orden cronologico'
assert (r4.index.to_series().diff().dropna() == pd.Timedelta(days=1)).all(), \
    'No puede haber huecos: la diferencia entre fechas consecutivas debe ser 1 dia'

_por_dia = {}
for _t in VENTAS.itertuples(index=False):
    k = _t.fecha.date()
    d = _por_dia.setdefault(k, [0, 0])
    d[0] += _t.importe; d[1] += 1

_sin_op = [d for d in r4.index if d.date() not in _por_dia]
assert len(_sin_op) == 7, f'Deben aparecer 7 dias sin operacion y aparecieron {len(_sin_op)}'
for _d in _sin_op:
    assert float(r4.loc[_d, 'importe']) == 0.0, \
        f'{_d.date()}: dia sin pedidos, importe debe ser 0 y no NaN'
    assert int(r4.loc[_d, 'pedidos']) == 0, f'{_d.date()}: pedidos debe ser 0'
for _k, (_imp, _n) in _por_dia.items():
    _ts = pd.Timestamp(_k)
    assert float(r4.loc[_ts, 'importe']) == float(_imp), \
        f'{_k}: importe deberia ser {_imp}'
    assert int(r4.loc[_ts, 'pedidos']) == _n, f'{_k}: pedidos deberia ser {_n}'
assert pd.api.types.is_integer_dtype(r4['pedidos']), \
    f'pedidos debe ser entero y quedo en {r4["pedidos"].dtype}'

assert r4['media7'].isna().sum() == 6, \
    f'Solo los primeros 6 dias pueden ser NaN en media7 (ventana de 7 incompleta), ' \
    f'y hay {r4["media7"].isna().sum()}. Revisa min_periods'
assert r4['media7'].iloc[:6].isna().all(), 'Los NaN de media7 van al principio, no al final'
_esp7 = r4['importe'].iloc[0:7].mean()
assert abs(float(r4['media7'].iloc[6]) - _esp7) < 1e-6, \
    f'media7 del dia 7 es el promedio de los dias 1 a 7 = {_esp7:.2f}, ' \
    'la ventana incluye el dia actual'
assert abs(float(r4['media7'].iloc[30]) - r4['importe'].iloc[24:31].mean()) < 1e-6, \
    'media7 de cada dia promedia ese dia y los 6 anteriores'

assert r4['indice'].isna().sum() == 6, 'indice es NaN exactamente donde media7 es NaN'
_ok = r4['media7'].notna()
assert np.allclose(r4.loc[_ok, 'indice'],
                   (r4.loc[_ok, 'importe'] / r4.loc[_ok, 'media7']).round(3)), \
    'indice = importe / media7 redondeado a 3 decimales'

# --- 4b ---
_picos = dias_pico(r4, 1.8)
assert isinstance(_picos, list), '4b debe devolver una lista'
assert all(isinstance(x, str) for x in _picos), \
    f'Cada fecha debe ser un str "YYYY-MM-DD"; te dio {type(_picos[0]).__name__}'
_esp = [d.strftime('%Y-%m-%d') for d in r4.index[r4['indice'] > 1.8]]
assert _picos == _esp, f'Con umbral 1.8 deberian ser {len(_esp)} dias: {_esp}'
assert _picos == sorted(_picos), 'En orden cronologico ascendente'
assert dias_pico(r4, 99.0) == [], 'Con un umbral imposible la lista debe quedar vacia'
assert len(dias_pico(r4, 1.0)) > len(_picos), 'Bajar el umbral debe traer mas dias'
assert all(x not in dias_pico(r4, 0.0) for x in
           r4.index[r4['indice'].isna()].strftime('%Y-%m-%d')), \
    'Un dia con indice NaN no puede salir como pico ni con umbral 0'

print('Reto 4 resuelto.')
print(f'{len(_picos)} dias por encima de 1.8 veces su media movil: {_picos}')
print()
print(r4.loc['2026-06-05':'2026-06-14'].round(2))

> **Si te sobra tiempo:** cambia la media movil por una **mediana** movil y mira cuales
> dias dejan de ser pico. La media se contagia del pico anterior porque el pico entra en
> su propia ventana los 7 dias siguientes; la mediana no. Ese detalle decide si tu
> deteccion sirve o solo se persigue la cola.

---
## Reto abierto - sin verificacion automatica

Los cuatro anteriores tenian una respuesta correcta. Este no.

**El encargo:** el gerente comercial te dice, textual, *"necesito saber a quien llamar
esta semana"*. No sabe que preguntar y no va a leer una tabla de 196 filas.

Escribe `clientes_para_llamar(ventas, clientes)` que imprima una lista corta y priorizada
de clientes, con **una razon en texto por cada uno**. Ideas, no todas buenas:

- Clientes que compraban seguido y llevan varias semanas sin pedir
- Clientes cuyo importe de julio cayo fuerte contra el de junio
- Clientes concentrados en una sola linea, que podrian comprar otra
- Clientes del maestro que nunca han comprado
- Los que compran mucho volumen pero con ticket bajo

Restricciones que hacen la diferencia:
- La lista debe tener **como maximo 8 clientes**. Priorizar es descartar.
- Cada uno lleva el **nombre** del cliente, no el `id_cliente`. El gerente no sabe que es C07.
- Si un criterio tuyo se apoya en un cliente que no esta en el maestro, decide que hacer
  con el y **deja escrita la decision**.

Se juzga el **criterio**: que decidiste mirar, por que eso y no otra cosa, y si el
informe le sirve a alguien que no programa.

In [ ]:
def clientes_para_llamar(ventas, clientes):
    # tu codigo aqui
    pass


clientes_para_llamar(VENTAS, CLIENTES)

**Tus decisiones de diseno** (escribe aqui que miraste, por que, y que dejaste por fuera):

1.
2.
3.

---
## Como se conecta con la Actividad 2

La **Actividad 2 (25%)** se llama *"Analisis de un Dataset con
Python: Limpieza y Transformacion"*. Pide un dataset publico en CSV cargado con pandas,
imputacion de faltantes (media, mediana o KNN), tratamiento de duplicados, valores fuera
de rango y outliers con **Z-Score o IQR**, normalizacion con **MinMaxScaler o
StandardScaler**, y sobre todo **funciones reutilizables para cada paso**. Se entrega
`.ipynb` mas PDF.

Lo que hiciste hoy no es un calentamiento: es la mitad del trabajo.

| Lo que pide la actividad | Lo que ya tienes de esta sesion |
|---|---|
| Funciones reutilizables por paso | Los cuatro retos son funciones con contrato y verificacion. Ese es exactamente el patron que pide la rubrica, y casi nadie lo entrega asi |
| Detectar valores faltantes | Reto 2: los `NaN` que aparecen tras un `left` o un `outer` no son "datos sucios", son la evidencia de que el cruce no casa. Saber de donde salio el `NaN` decide si se imputa o se investiga |
| Detectar outliers | Reto 4: `indice > umbral` es deteccion de atipicos con una linea base movil. Z-Score e IQR son la misma idea con otro estadistico. Ya sabes por que la ventana importa |
| Duplicados y consistencia | Reto 2: el maestro no tenia ids repetidos y por eso `n_left == len(ventas)`. Si los tuviera, el left **multiplica filas** y el total de ventas se infla. Ese es el duplicado que de verdad hace dano |
| Transformar sin perder filas | Reto 3a: `transform` conserva la forma. Escalar una variable con MinMax o StandardScaler es la misma operacion, y si la variable se escala **dentro de cada grupo**, es literalmente el mismo `transform` |
| Estructura del dataset | Reto 3b: `pivot_table` y `melt` son ancho contra largo. La mayoria de los datasets publicos vienen en el formato equivocado para lo que uno quiere hacer |

Lo que **no** viste hoy y toca para la actividad, que es lo de la sesion 4:

- Cargar desde archivo (`read_csv` con sus encodings, separadores y `na_values`) y los
  otros formatos que menciona la ruta: JSON y HDF5.
- Imputacion propiamente dicha: `fillna` con media o mediana, y `KNNImputer` de
  scikit-learn.
- Escalado con `MinMaxScaler` / `StandardScaler`.
- Duplicados con `duplicated` y `drop_duplicates`.

Un aviso que vale puntos: la rubrica premia **funciones reutilizables**, no un notebook
de celdas sueltas. Si organizas la entrega como una serie de funciones con nombre claro
que reciben un DataFrame y devuelven uno nuevo, ya estas por encima de la media. Y
documenta cada decision de limpieza: imputar y no decir con que criterio se castiga.

---
## Para terminar

1. Ejecuta el notebook completo de arriba abajo y confirma que las cuatro verificaciones pasan.
2. Guardalo **con las salidas visibles** (Archivo > Guardar).
3. Si quieres, subelo al aula (opcional, no se califica) y lo reviso y te comento por escrito.

Si terminaste temprano: toma `r3` (el DataFrame con `pct_linea` y `rank_linea`) y arma
un solo cuadro que responda "que producto sostiene cada linea". Un producto que es rank 1
en su linea pero aporta el 3% no sostiene nada; uno que aporta el 40% es un riesgo de
concentracion. Esa lectura es la que se va a pedir en la fase de visualizacion del Proyecto (Actividad 3) cuando toque
interpretar graficos, no solo dibujarlos.

**Proxima sesion:** limpieza y transformacion. La via avanzada arranca donde esta termina.

*Dudas: consulta al profesor*